In [ ]:
import os
import asyncio
from tenacity import retry, stop_after_attempt, wait_random_exponential, retry_if_exception
from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools.google_search_tool import google_search
from google.genai import types

# 1. SETUP ENVIRONMENT
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"
os.environ["GOOGLE_API_KEY"] = "xx" # Replace with your key
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"

# 2. DEFINE AGENTS
# Researcher uses the live search tool
search_specialist = LlmAgent(
    name="SearchSpecialist",
    model="gemini-2.5-flash-lite",
    instruction="""Find  top gainers/losers in the Indian stock market. 
    Use the Google Search tool. Focus on NSE and Moneycontrol data.""",
    tools=[google_search],
    output_key="market_data"
)

# Editor summarizes findings
editor = LlmAgent(
    name="Editor",
    model="gemini-2.5-flash-lite",
    instruction="Summarize the stock data found in {market_data} into a clear executive brief."
)

# The Sequential Team
my_team = SequentialAgent(
    name="TechInsightsTeam",
    sub_agents=[search_specialist, editor]
)

# 3. DEFINE RETRY LOGIC
# We specifically want to retry if we see a '429' in the error message
def is_rate_limit_error(exception):
    return "429" in str(exception) or "RESOURCE_EXHAUSTED" in str(exception)

@retry(
    retry=retry_if_exception(is_rate_limit_error),
    wait=wait_random_exponential(min=1, max=60), # Wait 1s, 2s, 4s... up to 60s
    stop=stop_after_attempt(5) # Give up after 5 tries
)
async def run_research_with_retry(query):
    session_service = InMemorySessionService()
    runner = Runner(
        agent=my_team, 
        app_name="StockApp", 
        session_service=session_service
    )
    
    session = await session_service.create_session(app_name="StockApp", user_id="user_1")
    content = types.Content(role="user", parts=[types.Part(text=query)])

    print(f"🚀 Attempting Research: {query}...")
    
    final_response = ""
    async for event in runner.run_async(
        new_message=content, 
        session_id=session.id,
        user_id="user_1"
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    final_response += part.text
    return final_response

# 4. EXECUTION
async def main():
    try:
        result = await run_research_with_retry("What are the best performing stocks in India today, Feb 13, 2026. Include actual gain and loss percentages ?")
        print("\n--- INDIAN MARKET BRIEFING ---\n")
        print(result)
    except Exception as e:
        print(f"\n❌ Permanent Failure: {e}")

await main()

🚀 Attempting Research: What are the best performing stocks in India today, Feb 13, 2026. Include actual gain and loss percentages ?...

--- INDIAN MARKET BRIEFING ---

On Friday, February 13, 2026, the Indian stock market experienced a significant downturn, with both the Sensex and Nifty indices falling by over 1%. This broad-based selloff was particularly pronounced in metal, IT, and commodity stocks, influenced by weak global market cues.

Here's a breakdown of the top gainers and losers on the NSE and BSE:

**Top Gainers:**

*   **Bajaj Finance Ltd:** Showed strong performance, with gains reported between 2.57% and 3.11% across different sources.
*   **Eicher Motors Ltd:** Also among the top gainers, with percentages ranging from 1.54% to 2.21%.
*   **SBI Life Insurance Company Ltd:** Reported gains between 0.12% and 0.84%.
*   **State Bank of India (SBI):** Saw gains of around 0.52%.
*   **Apollo Hospitals Enterprise Ltd:** Recorded gains of approximately 0.05% to 0.69%.
*   **Cipl